# RAG Project — NBA Player Q&A System (RAPTOR Statistics)

**Deep Learning — Bsc Data Science for Responsible Business — Centrale Lyon 2025-2026**

---

This notebook implements a complete **Retrieval-Augmented Generation (RAG)** pipeline by combining:
- **ChromaDB**: vector database for efficient semantic retrieval
- **Sentence Transformers (HuggingFace)**: embedding models
- **HuggingFace LLM**: answer generation
- **Gradio**: conversational user interface

**Dataset**: `538_historical_RAPTOR_by_player.csv` — Advanced NBA statistics (RAPTOR) by player and season since 1977.

## Table of Contents
1. Installation & Imports
2. Dataset Exploration
3. Document Preparation
4. Embedding Generation
5. ChromaDB Vector Store
6. HuggingFace LLM Integration
7. Complete RAG Pipeline
8. Evaluation & Optimization
9. Gradio Interface

---
## 1. Installation & Imports

In [1]:
# Install required dependencies
!pip install transformers chromadb sentence-transformers huggingface-hub \
             langchain_community langchain-text-splitters pypdf \
             gradio tqdm pandas matplotlib seaborn scikit-learn accelerate

  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached chromadb-1.5.8-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached sentence_transformers-5.4.1-py3-none-any.whl.metadata (17 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached gradio-6.13.0-py3-none-any.whl.metadata (17 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.41.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.41.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached kubernetes-35.0.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached langchain_core-1.3.0-py

ERROR: Could not install packages due to an OSError: [WinError 32] Le processus ne peut pas accéder au fichier car ce fichier est utilisé par un autre processus: 'c:\\Cour\\year_2\\Deep learning\\deep_learning_2526-main\\deep_learning_2526-main\\.venv\\Lib\\site-packages\\kubernetes\\client\\api\\resource_v1beta2_api.py'
Check the permissions.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

print("Imports OK")

---
## 2. Dataset Exploration

The **RAPTOR** (Robust Algorithm using Player Tracking and On/Off Ratings) dataset from FiveThirtyEight contains advanced metrics for evaluating NBA player performance. Each row represents one player's statistics for one season.

In [ ]:
df = pd.read_csv('538_historical_RAPTOR_by_player.csv')

print(f"Shape    : {df.shape}")
print(f"Columns  : {list(df.columns)}")
print(f"Seasons  : {df['season'].min()} → {df['season'].max()}")
print(f"Unique players : {df['player_name'].nunique()}")
df.head(5)

In [ ]:
# Statistical summary of RAPTOR metrics
cols_raptor = ['raptor_offense', 'raptor_defense', 'raptor_total', 'war_total']
df[cols_raptor].describe().round(3)

In [ ]:
# Missing values
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# Visualization 1: RAPTOR distributions and trends
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['raptor_total'].dropna(), bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Total RAPTOR')
axes[0].set_xlabel('RAPTOR Total')
axes[0].axvline(0, color='red', linestyle='--', label='League average')
axes[0].legend()

axes[1].scatter(df['raptor_offense'], df['raptor_defense'], alpha=0.3, s=5, color='steelblue')
axes[1].set_title('Offense vs Defense RAPTOR')
axes[1].set_xlabel('RAPTOR Offense')
axes[1].set_ylabel('RAPTOR Defense')
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5)

season_avg = df.groupby('season')['raptor_total'].mean()
axes[2].plot(season_avg.index, season_avg.values, color='steelblue', marker='o', markersize=3)
axes[2].set_title('Average RAPTOR Total by Season')
axes[2].set_xlabel('Season')
axes[2].set_ylabel('Average RAPTOR Total')
axes[2].axhline(0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 players of all time by cumulative WAR
top_players_war = (
    df.groupby('player_name')['war_total']
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top_players_war.index[::-1], top_players_war.values[::-1], color='steelblue')
ax.set_title('Top 15 Players — Cumulative Career WAR', fontsize=13)
ax.set_xlabel('Cumulative WAR Total')
for bar, val in zip(bars, top_players_war.values[::-1]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 single-season RAPTOR performances
top_season = (
    df.nlargest(10, 'raptor_total')[['player_name', 'season', 'raptor_offense', 'raptor_defense', 'raptor_total', 'war_total']]
    .reset_index(drop=True)
)
print("Top 10 single-season RAPTOR performances:")
top_season

---
## 3. Document Preparation

To build the RAG knowledge base, we convert tabular data into structured text documents. Each document describes one player's statistics for a given season, using qualitative labels to make the text semantically rich and searchable.

In [ ]:
def classify_raptor(value):
    """Maps a RAPTOR score to a qualitative performance label."""
    if value >= 6:
        return 'exceptional'
    elif value >= 3:
        return 'very good'
    elif value >= 1:
        return 'good'
    elif value >= -1:
        return 'average'
    elif value >= -3:
        return 'below average'
    else:
        return 'poor'


def classify_war(value):
    """Maps a WAR score to a qualitative impact label."""
    if value >= 8:
        return 'MVP candidate'
    elif value >= 4:
        return 'All-Star caliber'
    elif value >= 1:
        return 'solid starter'
    elif value >= 0:
        return 'useful rotation player'
    else:
        return 'negative impact'


def row_to_document(row):
    """Converts one DataFrame row into a structured text document."""
    doc = (
        f"{row['player_name']} during the {int(row['season'])} NBA season: "
        f"offensive RAPTOR of {row['raptor_offense']:.2f} ({classify_raptor(row['raptor_offense'])}), "
        f"defensive RAPTOR of {row['raptor_defense']:.2f} ({classify_raptor(row['raptor_defense'])}), "
        f"total RAPTOR of {row['raptor_total']:.2f} ({classify_raptor(row['raptor_total'])}). "
        f"Total WAR of {row['war_total']:.2f} ({classify_war(row['war_total'])}). "
        f"Possessions played: {int(row['poss'])}, minutes played: {int(row['mp'])}. "
        f"PREDATOR total score: {row['predator_total']:.2f}. "
        f"Pace impact: {row['pace_impact']:.3f}."
    )
    return doc


# Drop rows with missing values on key columns
df_clean = df.dropna(subset=['raptor_offense', 'raptor_defense', 'raptor_total', 'war_total']).copy()

# Generate all documents
documents = [row_to_document(row) for _, row in df_clean.iterrows()]

# Metadata associated with each document
metadatas = [
    {
        'player_name': row['player_name'],
        'season': int(row['season']),
        'raptor_total': float(round(row['raptor_total'], 3)),
        'war_total': float(round(row['war_total'], 3))
    }
    for _, row in df_clean.iterrows()
]

ids = [f"doc_{i}" for i in range(len(documents))]

print(f"Total documents created: {len(documents)}")
print("\nExample document:")
print(documents[10])

In [ ]:
# Document length statistics
lengths = [len(doc) for doc in documents]
print(f"Average length : {np.mean(lengths):.0f} characters")
print(f"Min / Max      : {min(lengths)} / {max(lengths)} characters")

plt.figure(figsize=(8, 3))
plt.hist(lengths, bins=40, color='steelblue', edgecolor='white')
plt.title('Document Length Distribution')
plt.xlabel('Number of characters')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

---
## 4. Embedding Generation

We compare two HuggingFace embedding models:
- **`all-MiniLM-L6-v2`**: lightweight, fast, 384 dimensions
- **`all-mpnet-base-v2`**: more powerful, 768 dimensions

Embeddings transform text into dense numerical vectors that capture semantic meaning, enabling similarity-based retrieval.

In [ ]:
# Load both embedding models
print("Loading model 1: all-MiniLM-L6-v2 ...")
model_minilm = SentenceTransformer('all-MiniLM-L6-v2')

print("Loading model 2: all-mpnet-base-v2 ...")
model_mpnet = SentenceTransformer('all-mpnet-base-v2')

print("Models loaded.")
print(f"  MiniLM  → embedding dimension: {model_minilm.get_sentence_embedding_dimension()}")
print(f"  MPNet   → embedding dimension: {model_mpnet.get_sentence_embedding_dimension()}")

In [ ]:
# Speed benchmark on a sample of 500 documents
SAMPLE = min(500, len(documents))
sample_docs = documents[:SAMPLE]

print(f"Benchmark on {SAMPLE} documents...")

# MiniLM
t0 = time.time()
emb_sample_minilm = model_minilm.encode(sample_docs, batch_size=64,
                                         normalize_embeddings=True, show_progress_bar=True)
t_minilm = time.time() - t0

# MPNet
t0 = time.time()
emb_sample_mpnet = model_mpnet.encode(sample_docs, batch_size=32,
                                       normalize_embeddings=True, show_progress_bar=True)
t_mpnet = time.time() - t0

print(f"\nMiniLM time : {t_minilm:.2f}s  ({t_minilm/SAMPLE*1000:.1f} ms/doc)")
print(f"MPNet time  : {t_mpnet:.2f}s  ({t_mpnet/SAMPLE*1000:.1f} ms/doc)")
print(f"Speed ratio : MPNet is {t_mpnet/t_minilm:.1f}x slower than MiniLM")

In [ ]:
# PCA visualization of MiniLM embeddings, colored by RAPTOR total
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(emb_sample_minilm)

raptor_vals = df_clean['raptor_total'].values[:SAMPLE]

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(coords[:, 0], coords[:, 1], c=raptor_vals, cmap='RdYlGn',
                alpha=0.6, s=10, vmin=-8, vmax=8)
plt.colorbar(sc, ax=ax, label='RAPTOR Total')
ax.set_title('PCA of MiniLM Embeddings (colored by RAPTOR Total)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.tight_layout()
plt.show()

In [ ]:
# Semantic similarity test between example sentences
test_sentences = [
    "Michael Jordan during the 1996 NBA season",
    "LeBron James during the 2013 NBA season",
    "a player with an exceptional total RAPTOR score",
    "a player with poor defensive skills"
]

emb_test = model_minilm.encode(test_sentences, normalize_embeddings=True)
sim_matrix = cosine_similarity(emb_test)

fig, ax = plt.subplots(figsize=(6, 5))
labels = [s[:40] + '...' if len(s) > 40 else s for s in test_sentences]
sns.heatmap(sim_matrix, annot=True, fmt='.3f', xticklabels=labels, yticklabels=labels,
            cmap='Blues', ax=ax, vmin=0, vmax=1)
ax.set_title('Cosine Similarity Matrix (MiniLM)')
plt.tight_layout()
plt.show()

---
## 5. ChromaDB Vector Store

ChromaDB stores documents and their embeddings, enabling fast and accurate semantic retrieval using approximate nearest-neighbor search (HNSW index).

In [ ]:
# Initialize ChromaDB persistent client
client = PersistentClient(path="./raptor_chroma_db")

# --- MiniLM Collection ---
COLLECTION_MINILM = "raptor_minilm"
try:
    client.delete_collection(name=COLLECTION_MINILM)
except:
    pass
collection_minilm = client.create_collection(name=COLLECTION_MINILM,
                                              metadata={"hnsw:space": "cosine"})

# --- MPNet Collection ---
COLLECTION_MPNET = "raptor_mpnet"
try:
    client.delete_collection(name=COLLECTION_MPNET)
except:
    pass
collection_mpnet = client.create_collection(name=COLLECTION_MPNET,
                                             metadata={"hnsw:space": "cosine"})

print("ChromaDB collections created.")

In [ ]:
# Generate all embeddings with MiniLM
print("Generating all MiniLM embeddings...")
t0 = time.time()
all_embeddings_minilm = model_minilm.encode(
    documents, batch_size=128, normalize_embeddings=True, show_progress_bar=True
)
print(f"Done in {time.time()-t0:.1f}s")

In [ ]:
# Index into ChromaDB in batches
BATCH_SIZE = 500
n = len(documents)

print(f"Indexing {n} documents into ChromaDB (MiniLM)...")
for start in tqdm(range(0, n, BATCH_SIZE)):
    end = min(start + BATCH_SIZE, n)
    collection_minilm.add(
        documents=documents[start:end],
        embeddings=all_embeddings_minilm[start:end].tolist(),
        metadatas=metadatas[start:end],
        ids=ids[start:end]
    )

print(f"MiniLM collection: {collection_minilm.count()} documents indexed.")

In [ ]:
# Generate all embeddings with MPNet and index
print("Generating all MPNet embeddings...")
t0 = time.time()
all_embeddings_mpnet = model_mpnet.encode(
    documents, batch_size=64, normalize_embeddings=True, show_progress_bar=True
)
print(f"Done in {time.time()-t0:.1f}s")

print("Indexing into ChromaDB (MPNet)...")
for start in tqdm(range(0, n, BATCH_SIZE)):
    end = min(start + BATCH_SIZE, n)
    collection_mpnet.add(
        documents=documents[start:end],
        embeddings=all_embeddings_mpnet[start:end].tolist(),
        metadatas=metadatas[start:end],
        ids=ids[start:end]
    )

print(f"MPNet collection: {collection_mpnet.count()} documents indexed.")

In [ ]:
# Retrieval test
def retrieve(question, collection, model, n_results=3):
    """Retrieves the n_results most relevant documents for a given query."""
    query_emb = model.encode([question], normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings=query_emb,
        n_results=n_results,
        include=['documents', 'metadatas', 'distances']
    )
    return results

# Test retrieval
test_query = "Who was the best offensive player in the NBA in 1996?"
results = retrieve(test_query, collection_minilm, model_minilm, n_results=3)

print(f"Query: {test_query}\n")
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
)):
    print(f"[{i+1}] Similarity score: {1 - dist:.4f}")
    print(f"     {doc[:200]}...\n")

---
## 6. HuggingFace LLM Integration

We use a HuggingFace LLM for answer generation and experiment with multiple prompt templates to assess their impact on response quality.

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = 0 if torch.cuda.is_available() else -1
print(f"Device: {'GPU (CUDA)' if device == 0 else 'CPU'}")

In [ ]:
# Load the LLM — google/flan-t5-base (lightweight Seq2Seq, no GPU required)
# Alternatives: google/flan-t5-large, HuggingFaceH4/zephyr-7b-beta, mistralai/Mistral-7B-Instruct-v0.1

MODEL_NAME = "google/flan-t5-base"
print(f"Loading LLM: {MODEL_NAME} ...")

llm_pipeline = pipeline(
    "text2text-generation",
    model=MODEL_NAME,
    device=device,
    max_new_tokens=256
)

print("LLM loaded.")

In [ ]:
# =============================================
# Prompt Templates — Experimentation
# =============================================

PROMPT_TEMPLATES = {
    "standard": """\
Use the information enclosed in <context> tags to provide an answer to the question \
enclosed in <question> tags.
<context>
{context}
</context>
<question>
{question}
</question>
Answer:""",

    "concise": """\
Answer the question based on the context below. Be concise.

Context: {context}

Question: {question}

Short answer:""",

    "expert": """\
You are an NBA analytics expert. Using only the statistics provided below, \
answer the question accurately.

Statistics:
{context}

Question: {question}

Expert analysis:""",

    "chain_of_thought": """\
Based on the NBA data provided, answer the question step by step.

Data:
{context}

Question: {question}

Let me analyze the data step by step:"""
}

print(f"{len(PROMPT_TEMPLATES)} prompt templates defined: {list(PROMPT_TEMPLATES.keys())}")

In [ ]:
def build_context(results, max_chars=1500):
    """Builds the context string from retrieved documents."""
    context_lines = []
    total = 0
    for doc in results['documents'][0]:
        if total + len(doc) > max_chars:
            break
        context_lines.append(doc)
        total += len(doc)
    return "\n".join(context_lines)


def generate_answer(question, collection, embed_model, llm, template_name='standard',
                    n_results=3, max_new_tokens=200):
    """Full RAG pipeline: retrieval → prompt construction → LLM generation."""
    # 1. Retrieval
    t0 = time.time()
    results = retrieve(question, collection, embed_model, n_results=n_results)
    t_retrieval = time.time() - t0

    # 2. Context and prompt construction
    context = build_context(results)
    template = PROMPT_TEMPLATES[template_name]
    prompt = template.format(context=context, question=question)

    # 3. Generation
    t0 = time.time()
    output = llm(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    t_generation = time.time() - t0

    answer = output[0]['generated_text'].strip()

    return {
        'question': question,
        'answer': answer,
        'context': context,
        'retrieved_docs': results['documents'][0],
        'distances': results['distances'][0],
        'template': template_name,
        't_retrieval_ms': round(t_retrieval * 1000, 1),
        't_generation_ms': round(t_generation * 1000, 1),
        't_total_ms': round((t_retrieval + t_generation) * 1000, 1)
    }

In [ ]:
# Test the full RAG pipeline
test_questions = [
    "Who had the highest RAPTOR total in the 1996 NBA season?",
    "What was LeBron James best offensive season according to RAPTOR?",
    "Which player had the most WAR in NBA history based on the data?"
]

for q in test_questions:
    result = generate_answer(q, collection_minilm, model_minilm, llm_pipeline,
                              template_name='standard', n_results=3)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"   [Retrieval: {result['t_retrieval_ms']}ms | Generation: {result['t_generation_ms']}ms]")
    print()

---
## 7. Complete RAG Pipeline

The `RAPTORRag` class encapsulates the full pipeline and adds usage tracking.

In [ ]:
class RAPTORRag:
    """
    Complete RAG system for NBA RAPTOR statistics.

    Pipeline: Query → Embedding → ChromaDB Retrieval → Prompt → LLM → Response
    """

    def __init__(self, collection, embed_model, llm,
                 default_template='standard', n_results=3):
        self.collection = collection
        self.embed_model = embed_model
        self.llm = llm
        self.default_template = default_template
        self.n_results = n_results
        self.history = []

    def ask(self, question, template_name=None, n_results=None, verbose=False):
        """Ask a question and return the RAG-generated answer."""
        template = template_name or self.default_template
        k = n_results or self.n_results

        result = generate_answer(
            question, self.collection, self.embed_model,
            self.llm, template_name=template, n_results=k
        )
        self.history.append(result)

        if verbose:
            print(f"Question : {result['question']}")
            print(f"Answer   : {result['answer']}")
            print(f"Template : {template}  |  k={k}")
            print(f"Latency  : {result['t_total_ms']}ms total "
                  f"(retrieval={result['t_retrieval_ms']}ms, "
                  f"generation={result['t_generation_ms']}ms)")
            print("Retrieved documents:")
            for i, (doc, dist) in enumerate(zip(result['retrieved_docs'], result['distances'])):
                print(f"  [{i+1}] sim={1-dist:.4f}  {doc[:120]}...")

        return result['answer']

    def get_stats(self):
        """Returns usage statistics for the system."""
        if not self.history:
            return {}
        latencies = [h['t_total_ms'] for h in self.history]
        return {
            'n_queries': len(self.history),
            'avg_latency_ms': round(np.mean(latencies), 1),
            'min_latency_ms': round(np.min(latencies), 1),
            'max_latency_ms': round(np.max(latencies), 1),
        }


# Instantiate the main RAG system
rag = RAPTORRag(
    collection=collection_minilm,
    embed_model=model_minilm,
    llm=llm_pipeline,
    default_template='standard',
    n_results=3
)

print("RAG system ready.")

In [ ]:
# Full demonstration with detailed output
demo_questions = [
    "Who was the best defensive player in the NBA in 2004?",
    "What is Michael Jordan's RAPTOR total in 1988?",
    "Which players had a WAR greater than 10 in any single season?"
]

for q in demo_questions:
    print("=" * 70)
    rag.ask(q, verbose=True)
    print()

---
## 8. Evaluation & Optimization

### 8.1 Embedding Model Comparison

In [ ]:
# Evaluation set with ground-truth expectations
eval_set = [
    {"question": "What was Michael Jordan RAPTOR total in 1996?",
     "expected_player": "Michael Jordan", "expected_season": 1996},
    {"question": "LeBron James best WAR season statistics",
     "expected_player": "LeBron James", "expected_season": None},
    {"question": "Kareem Abdul-Jabbar offensive RAPTOR 1977",
     "expected_player": "Kareem Abdul-Jabbar", "expected_season": 1977},
    {"question": "Who had negative RAPTOR defense in 2000?",
     "expected_player": None, "expected_season": 2000},
    {"question": "Best pace impact player statistics",
     "expected_player": None, "expected_season": None},
]


def eval_retrieval(eval_set, collection, model, n_results=3):
    """Evaluates retrieval quality using Hit Rate @k."""
    hits = 0
    latencies = []

    for item in eval_set:
        t0 = time.time()
        results = retrieve(item['question'], collection, model, n_results=n_results)
        latencies.append((time.time() - t0) * 1000)

        retrieved_metas = results['metadatas'][0]
        hit = False

        for meta in retrieved_metas:
            player_match = (item['expected_player'] is None or
                            item['expected_player'].lower() in meta['player_name'].lower())
            season_match = (item['expected_season'] is None or
                            meta['season'] == item['expected_season'])
            if player_match and season_match:
                hit = True
                break

        hits += int(hit)

    return {
        'hit_rate': hits / len(eval_set),
        'avg_latency_ms': round(np.mean(latencies), 1),
        'n_eval': len(eval_set)
    }


print("Evaluating retrieval quality...")

metrics_minilm = eval_retrieval(eval_set, collection_minilm, model_minilm)
metrics_mpnet  = eval_retrieval(eval_set, collection_mpnet, model_mpnet)

print("\n=== Embedding Model Comparison ===")
print(f"{'Metric':<25} {'MiniLM-L6-v2':>15} {'MPNet-base-v2':>15}")
print("-" * 55)
print(f"{'Hit Rate @3':<25} {metrics_minilm['hit_rate']*100:>14.0f}% {metrics_mpnet['hit_rate']*100:>14.0f}%")
print(f"{'Avg Latency (ms)':<25} {metrics_minilm['avg_latency_ms']:>15.1f} {metrics_mpnet['avg_latency_ms']:>15.1f}")

In [ ]:
# Visualization: model comparison
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

models_names = ['MiniLM-L6-v2', 'MPNet-base-v2']
hit_rates = [metrics_minilm['hit_rate'], metrics_mpnet['hit_rate']]
latencies_vals = [metrics_minilm['avg_latency_ms'], metrics_mpnet['avg_latency_ms']]
colors = ['steelblue', 'darkorange']

axes[0].bar(models_names, [h*100 for h in hit_rates], color=colors)
axes[0].set_title('Hit Rate @3 (%)')
axes[0].set_ylabel('Hit Rate (%)')
axes[0].set_ylim(0, 105)
for i, v in enumerate(hit_rates):
    axes[0].text(i, v*100 + 1, f'{v*100:.0f}%', ha='center', fontweight='bold')

axes[1].bar(models_names, latencies_vals, color=colors)
axes[1].set_title('Retrieval Latency (ms)')
axes[1].set_ylabel('ms')
for i, v in enumerate(latencies_vals):
    axes[1].text(i, v + 0.5, f'{v:.1f}ms', ha='center', fontweight='bold')

plt.suptitle('Embedding Model Comparison', fontsize=13)
plt.tight_layout()
plt.show()

### 8.2 Prompt Template Comparison

In [ ]:
# Compare all 4 templates on the same question
test_q = "Who was the most dominant offensive player in the 2000 NBA season?"

print(f"Question: {test_q}\n")
print("=" * 70)

template_results = {}
for tname in PROMPT_TEMPLATES:
    result = generate_answer(test_q, collection_minilm, model_minilm,
                              llm_pipeline, template_name=tname, n_results=3)
    template_results[tname] = result
    print(f"[{tname.upper()}]")
    print(f"  Answer  : {result['answer']}")
    print(f"  Latency : {result['t_total_ms']}ms")
    print()

In [ ]:
# Latency breakdown by template
t_retrieval_by_template = [template_results[t]['t_retrieval_ms'] for t in PROMPT_TEMPLATES]
t_gen_by_template       = [template_results[t]['t_generation_ms'] for t in PROMPT_TEMPLATES]

x = np.arange(len(PROMPT_TEMPLATES))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
bars1 = ax.bar(x - width/2, t_retrieval_by_template, width, label='Retrieval', color='steelblue')
bars2 = ax.bar(x + width/2, t_gen_by_template, width, label='LLM Generation', color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(list(PROMPT_TEMPLATES.keys()))
ax.set_ylabel('Latency (ms)')
ax.set_title('Latency per Step and Prompt Template')
ax.legend()
plt.tight_layout()
plt.show()

### 8.3 Impact of k (number of retrieved documents)

In [ ]:
# Hit Rate as a function of k
k_values = [1, 2, 3, 5, 7, 10]
hit_rates_k = []

for k in k_values:
    m = eval_retrieval(eval_set, collection_minilm, model_minilm, n_results=k)
    hit_rates_k.append(m['hit_rate'])
    print(f"k={k:2d}  →  Hit Rate = {m['hit_rate']*100:.0f}%")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, [h*100 for h in hit_rates_k], marker='o', color='steelblue', linewidth=2)
ax.set_xlabel('k (number of retrieved documents)')
ax.set_ylabel('Hit Rate (%)')
ax.set_title('Hit Rate as a Function of k (MiniLM)')
ax.set_xticks(k_values)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

### 8.4 Latency Benchmark

In [ ]:
# Full latency benchmark across 10 varied queries
bench_questions = [
    "Who is the best player in 1990?",
    "LeBron James RAPTOR statistics in 2010",
    "Which player had negative war_total in 2005?",
    "Best defensive RAPTOR in NBA 2015",
    "Michael Jordan 1993 statistics",
    "Highest pace impact in the dataset",
    "Kobe Bryant best season according to RAPTOR",
    "Players with excellent predator total score",
    "Who played the most minutes in 2002?",
    "Best WAR player in the 1980s"
]

bench_times = []
for q in bench_questions:
    r = generate_answer(q, collection_minilm, model_minilm, llm_pipeline)
    bench_times.append(r['t_total_ms'])

print(f"\n=== Latency Benchmark (n={len(bench_questions)} queries) ===")
print(f"  Average latency : {np.mean(bench_times):.0f} ms")
print(f"  Min latency     : {np.min(bench_times):.0f} ms")
print(f"  Max latency     : {np.max(bench_times):.0f} ms")
print(f"  Std deviation   : {np.std(bench_times):.0f} ms")

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(len(bench_times)), bench_times, color='steelblue', edgecolor='white')
ax.axhline(np.mean(bench_times), color='red', linestyle='--',
           label=f'Average: {np.mean(bench_times):.0f}ms')
ax.set_xlabel('Query #')
ax.set_ylabel('Total latency (ms)')
ax.set_title('RAG System Latency per Query')
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Gradio Interface

A conversational web interface that lets users query the RAG system interactively, choose the prompt template, and adjust the number of retrieved documents.

In [ ]:
import gradio as gr

# Select the best embedding model based on evaluation metrics
best_collection = collection_minilm if metrics_minilm['hit_rate'] >= metrics_mpnet['hit_rate'] else collection_mpnet
best_model = model_minilm if best_collection is collection_minilm else model_mpnet
best_model_name = 'MiniLM-L6-v2' if best_collection is collection_minilm else 'MPNet-base-v2'

print(f"Best embedding model selected: {best_model_name}")


def rag_chat(message, history, template_choice, n_results_choice):
    """Main RAG chatbot function — returns answer enriched with source documents."""
    if not message.strip():
        return history, "", ""

    result = generate_answer(
        question=message,
        collection=best_collection,
        embed_model=best_model,
        llm=llm_pipeline,
        template_name=template_choice,
        n_results=int(n_results_choice)
    )

    answer = result['answer']
    history.append((message, answer))

    # Format retrieved sources
    sources_text = "**Retrieved Sources:**\n"
    for i, (doc, dist) in enumerate(zip(result['retrieved_docs'], result['distances'])):
        sim = 1 - dist
        sources_text += f"\n**[{i+1}]** (Similarity: {sim:.3f})\n{doc[:250]}...\n"

    # Performance info
    perf_text = (
        f"Retrieval: **{result['t_retrieval_ms']}ms** | "
        f"Generation: **{result['t_generation_ms']}ms** | "
        f"Total: **{result['t_total_ms']}ms** | "
        f"Template: **{template_choice}** | k=**{n_results_choice}**"
    )

    return history, sources_text, perf_text


def clear_chat():
    return [], "", ""


# ===== Gradio Interface =====
with gr.Blocks(theme=gr.themes.Soft(), title="RAG NBA RAPTOR") as demo:

    gr.Markdown("""
    # 🏀 RAG System — NBA RAPTOR Statistics
    **Ask questions about advanced NBA player statistics (1977 – present)**

    Powered by: **ChromaDB** + **Sentence Transformers** + **Flan-T5**
    """)

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Conversation", height=400, bubble_full_width=False)
            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="E.g.: Who had the highest RAPTOR in 1996? What are LeBron's stats in 2013?",
                    label="Your question",
                    scale=4
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)

            clear_btn = gr.Button("Clear conversation", variant="secondary")
            perf_info = gr.Markdown(label="Performance")

        with gr.Column(scale=1):
            gr.Markdown("### Settings")
            template_dropdown = gr.Dropdown(
                choices=list(PROMPT_TEMPLATES.keys()),
                value='standard',
                label="Prompt Template"
            )
            n_results_slider = gr.Slider(
                minimum=1, maximum=10, value=3, step=1,
                label="Number of retrieved documents (k)"
            )
            sources_box = gr.Markdown(label="Sources")

    gr.Markdown("""
    ---
    ### Example Questions
    - *Who had the highest RAPTOR total in the 1996 season?*
    - *What is Michael Jordan's offensive RAPTOR in 1991?*
    - *Which player had the best WAR total ever?*
    - *Who was the best defensive player in 2004?*
    - *LeBron James statistics in 2013*
    """)

    gr.Examples(
        examples=[
            ["Who had the highest RAPTOR total in the 1996 NBA season?"],
            ["What is Michael Jordan offensive RAPTOR in 1991?"],
            ["Which player had the best WAR total ever?"],
            ["Who was the best defensive player in 2004?"],
            ["LeBron James statistics in 2013"]
        ],
        inputs=[msg_input]
    )

    send_btn.click(
        fn=rag_chat,
        inputs=[msg_input, chatbot, template_dropdown, n_results_slider],
        outputs=[chatbot, sources_box, perf_info]
    ).then(lambda: "", outputs=msg_input)

    msg_input.submit(
        fn=rag_chat,
        inputs=[msg_input, chatbot, template_dropdown, n_results_slider],
        outputs=[chatbot, sources_box, perf_info]
    ).then(lambda: "", outputs=msg_input)

    clear_btn.click(fn=clear_chat, outputs=[chatbot, sources_box, perf_info])

print("Gradio interface ready. Launching...")
demo.launch(share=False)

---
## Conclusion

This project implemented a **complete RAG system** on the NBA RAPTOR dataset:

| Component | Choice | Justification |
|-----------|--------|---------------|
| **Dataset** | FiveThirtyEight RAPTOR | Rich, real-world NBA data |
| **Documents** | Text descriptions per player/season | Structured conversion from CSV to natural language |
| **Embedding 1** | `all-MiniLM-L6-v2` | Lightweight, fast, excellent performance/speed ratio |
| **Embedding 2** | `all-mpnet-base-v2` | More powerful, compared in evaluation |
| **Vector DB** | ChromaDB (persistent) | Open-source, easy to use, HNSW indexing |
| **LLM** | `google/flan-t5-base` | Seq2Seq, lightweight, usable without GPU |
| **Interface** | Gradio | Simple conversational web interface |

### Key Takeaways
- **Document quality** (structuring tabular data as text) is critical for retrieval relevance
- The **embedding model** choice significantly impacts both hit rate and latency
- The **prompt template** has a measurable effect on the quality of generated answers
- The parameter **k** offers a precision vs. context trade-off
- ChromaDB provides sufficient persistence and performance for this use case

### Possible Improvements
- Use a more powerful LLM (Zephyr-7B, Mistral-7B) for higher-quality answers
- Implement **re-ranking** of retrieved documents
- Add **conversational memory** to handle follow-up questions
- Enrich documents with computed statistics (rankings, comparisons across seasons)